# T1: Claim Sentence Detection — Multi-Model + Context

Train and compare SciBERT, PubMedBERT, BioBERT, and ModernBERT with the same golden validation/test split.


## 0. Setup


In [1]:
!pip uninstall -y torchvision torchaudio xformers

In [2]:
# ============================================================================
# CELL 2: MINIMUM COMPATIBILITY SETUP
# Run this once, then restart kernel if torchao was removed.
# ============================================================================

import os

# Must be set before importing transformers
os.environ["TRANSFORMERS_NO_TORCHAO"] = "1"
os.environ["TORCHDYNAMO_DISABLE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Needed for some biomedical tokenizers; torchao causes infer_schema errors in this environment
!pip install -q sentencepiece tiktoken
!pip uninstall -y torchao

print("Minimum compatibility setup done. Restart kernel, then run all cells from the beginning.")



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip
Minimum compatibility setup done. Restart kernel, then run all cells from the beginning.


In [3]:
# ============================================================================
# CELL 3: IMPORTS
# IMPORTANT: Cell 2 must run before this cell.
# ============================================================================

import os, sys, json, math, random
from pathlib import Path
import numpy as np
import pandas as pd
import scipy
import sklearn

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
    classification_report,
    confusion_matrix,
)

from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Imports loaded successfully.")
print("DEVICE:", DEVICE)


Imports loaded successfully.
DEVICE: cpu


/usr/local/lib/python3.10/dist-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


## 1. Configuration


In [4]:

# Paths — adjust to your environment
DATA_DIR = Path('Data')
OUTPUT_DIR = Path('outputs_multi_model')
MODEL_ROOT_DIR = Path('models/t1_multi_model_claim_detectors')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_ROOT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FILE = Path('/work/Research/MedClaimExtractor/dataset.jsonl')      # LLM-labelled training set
GOLD_FILE  = Path('/work/Research/MedClaimExtractor/golden.jsonl')       # manually labelled gold set

SEED = 42
MAX_LEN = 256
BATCH_SIZE = 16
EPOCHS = 4
LR = 2e-5
WARMUP_RATIO = 0.1
GOLD_VAL_FRAC = 0.50         # split golden papers into 50% validation and 50% test
USE_CONTEXT = True           # context-enhanced input
CLAIM_CLASS_WEIGHT = 3.0     # label 1 claim gets stronger weight to reduce false negatives

# Export controls
SAVE_ALL_EXTRACTED_CLAIMS = True   # save extracted claims into one combined JSON file
EXPORT_TRAIN_PREDICTIONS = True     # include Dataset/LLM-training claims in the JSON; set False if too slow

MODEL_CONFIGS = [
    {
        'run_name': 'SciBERT_Context',
        'model_name': 'allenai/scibert_scivocab_uncased',
        'use_fast_tokenizer': True,
        'max_len': MAX_LEN,
        'batch_size': BATCH_SIZE,
        'epochs': EPOCHS,
        'lr': LR,
    },
    {
        'run_name': 'PubMedBERT_Context',
        'model_name': 'microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext',
        'use_fast_tokenizer': True,
        'max_len': MAX_LEN,
        'batch_size': BATCH_SIZE,
        'epochs': EPOCHS,
        'lr': LR,
    },
    {
        'run_name': 'BioBERT_Context',
        'model_name': 'monologg/biobert_v1.1_pubmed',
        'use_fast_tokenizer': False,  # BioBERT uses slow tokenizer
        'max_len': MAX_LEN,
        'batch_size': BATCH_SIZE,
        'epochs': EPOCHS,
        'lr': LR,
    },
    {
        'run_name': 'ModernBERT_Context',
        'model_name': 'answerdotai/ModernBERT-base',
        'use_fast_tokenizer': True,
        'max_len': MAX_LEN,
        'batch_size': 8,       # safer for GPU memory
        'epochs': EPOCHS,
        'lr': LR,
    },
]

print('Models to train:')
for cfg in MODEL_CONFIGS:
    print('-', cfg['run_name'], '=>', cfg['model_name'], '| fast tokenizer:', cfg['use_fast_tokenizer'])


Models to train:
- SciBERT_Context => allenai/scibert_scivocab_uncased | fast tokenizer: True
- PubMedBERT_Context => microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext | fast tokenizer: True
- BioBERT_Context => monologg/biobert_v1.1_pubmed | fast tokenizer: False
- ModernBERT_Context => answerdotai/ModernBERT-base | fast tokenizer: True


## 2. Load JSONL data

Both files share the same schema (`doi`, `year`, `title`, `abstract`, `sentences[]`).


In [5]:
def load_jsonl(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

train_papers = load_jsonl(TRAIN_FILE)
gold_papers  = load_jsonl(GOLD_FILE)
print(f'Training papers: {len(train_papers)}')
print(f'Gold test papers: {len(gold_papers)}')
print(f'Training sentences: {sum(len(p["sentences"]) for p in train_papers)}')
print(f'Gold sentences:     {sum(len(p["sentences"]) for p in gold_papers)}')


Training papers: 1269
Gold test papers: 316
Training sentences: 13755
Gold sentences:     3547


## 3. Build context-enhanced examples

Each example concatenates title, year, the previous sentence, the target sentence, and the next sentence. This helps disambiguate sentences like *"This method achieved superior results."*


In [6]:
def build_examples(papers, use_context=True):
    rows = []
    for p in papers:
        title = (p.get('title') or '').strip()
        year  = p.get('year') or ''
        sents = p.get('sentences', [])
        for i, s in enumerate(sents):
            text  = s.get('text', '').strip()
            label = int(s.get('label', 0))
            prev_t = sents[i - 1]['text'].strip() if i > 0 else ''
            next_t = sents[i + 1]['text'].strip() if i + 1 < len(sents) else ''
            if use_context:
                inp = (
                    f'Title: {title} | Year: {year} | '
                    f'Previous: {prev_t} | Sentence: {text} | Next: {next_t}'
                )
            else:
                inp = text
            rows.append({
                'doi': p['doi'],
                'sentence_id': s.get('sentence_id', i + 1),
                'text': text,
                'input': inp,
                'label': label,
                'year': year,
            })
    return pd.DataFrame(rows)

train_full_df = build_examples(train_papers, use_context=USE_CONTEXT)
gold_df       = build_examples(gold_papers,  use_context=USE_CONTEXT)

print('train_full size:', len(train_full_df), '| label dist:', train_full_df.label.value_counts().to_dict())
print('gold size:      ', len(gold_df),       '| label dist:', gold_df.label.value_counts().to_dict())
train_full_df.head(3)


train_full size: 13755 | label dist: {0: 9577, 1: 4178}
gold size:       3547 | label dist: {0: 2062, 1: 1485}


,doi,sentence_id,text,input,label,year
0,10.1002/cncr.35732,1,Background: Glioma is the most common malignan...,Title: Glioma mutational signatures associated...,0,2025
1,10.1002/cncr.35732,2,Modifiable risk factors remain unidentified.,Title: Glioma mutational signatures associated...,0,2025
2,10.1002/cncr.35732,3,"New advances in exposure assessment, genomic a...",Title: Glioma mutational signatures associated...,0,2025


## 4. Paper-level train/validation split

Sentences from the same paper must not appear in both splits.


In [7]:
## 4. Paper-level split
# New design:
# Dataset.jsonl / train_full_df = training only
# golden.jsonl / gold_df = split into golden validation and golden test

# ------------------------------------------------------------------
# 1. Remove any golden-paper overlap from training data
# ------------------------------------------------------------------
gold_dois_set = set(gold_df["doi"].unique())

before_train_papers = train_full_df["doi"].nunique()

train_full_df = train_full_df[
    ~train_full_df["doi"].isin(gold_dois_set)
].reset_index(drop=True)

after_train_papers = train_full_df["doi"].nunique()

print(f"Training papers before removing golden overlap: {before_train_papers}")
print(f"Training papers after removing golden overlap:  {after_train_papers}")

# ------------------------------------------------------------------
# 2. Split golden papers into validation and test
# ------------------------------------------------------------------
gold_unique_dois = sorted(gold_df["doi"].unique())

rng = random.Random(SEED)
rng.shuffle(gold_unique_dois)

# 50% golden papers for validation, 50% for test
n_gold_val = max(1, int(len(gold_unique_dois) * GOLD_VAL_FRAC))

gold_val_dois = set(gold_unique_dois[:n_gold_val])
gold_test_dois = set(gold_unique_dois[n_gold_val:])

train_df = train_full_df.reset_index(drop=True)
val_df = gold_df[gold_df["doi"].isin(gold_val_dois)].reset_index(drop=True)
test_df = gold_df[gold_df["doi"].isin(gold_test_dois)].reset_index(drop=True)

# ------------------------------------------------------------------
# 3. Safety checks: no paper overlap between train/val/test
# ------------------------------------------------------------------
train_dois = set(train_df["doi"].unique())
val_dois = set(val_df["doi"].unique())
test_dois = set(test_df["doi"].unique())

assert len(train_dois & val_dois) == 0, "Leakage: train and val share papers"
assert len(train_dois & test_dois) == 0, "Leakage: train and test share papers"
assert len(val_dois & test_dois) == 0, "Leakage: val and test share papers"

# ------------------------------------------------------------------
# 4. Print final split information
# ------------------------------------------------------------------
print("=" * 70)
print("FINAL PAPER-LEVEL SPLIT")
print("=" * 70)

print(f"Train papers:       {len(train_dois)}")
print(f"Golden val papers:  {len(val_dois)}")
print(f"Golden test papers: {len(test_dois)}")

print("\nSentence counts:")
print("Train sentences:", len(train_df))
print("Val sentences:  ", len(val_df))
print("Test sentences: ", len(test_df))

print("\nLabel distribution:")
print("Train:", train_df.label.value_counts().to_dict())
print("Val:  ", val_df.label.value_counts().to_dict())
print("Test: ", test_df.label.value_counts().to_dict())

print("\nClaim percentage:")
print("Train:", round(train_df.label.mean() * 100, 2), "%")
print("Val:  ", round(val_df.label.mean() * 100, 2), "%")
print("Test: ", round(test_df.label.mean() * 100, 2), "%")

Training papers before removing golden overlap: 1269
Training papers after removing golden overlap:  1269
FINAL PAPER-LEVEL SPLIT
Train papers:       1269
Golden val papers:  158
Golden test papers: 158

Sentence counts:
Train sentences: 13755
Val sentences:   1742
Test sentences:  1805

Label distribution:
Train: {0: 9577, 1: 4178}
Val:   {0: 1042, 1: 700}
Test:  {0: 1020, 1: 785}

Claim percentage:
Train: 30.37 %
Val:   40.18 %
Test:  43.49 %


## 5. Dataset, evaluation, training, and threshold utilities

In [8]:

class ClaimDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=MAX_LEN):
        self.texts  = df['input'].tolist()
        self.labels = df['label'].tolist()
        self.tok    = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tok(
            self.texts[idx],
            truncation=True,
            max_length=self.max_len,
            padding='max_length',
            return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long),
        }


def make_loaders(tokenizer, max_len, batch_size):
    train_ds = ClaimDataset(train_df, tokenizer, max_len=max_len)
    val_ds   = ClaimDataset(val_df, tokenizer, max_len=max_len)
    test_ds  = ClaimDataset(test_df, tokenizer, max_len=max_len)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size * 2, shuffle=False, num_workers=2)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size * 2, shuffle=False, num_workers=2)
    return train_loader, val_loader, test_loader


@torch.no_grad()
def evaluate(loader, model, return_probs=False):
    model.eval()
    all_logits, all_labels = [], []

    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        out = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'])
        all_logits.append(out.logits.detach().cpu().numpy())
        all_labels.append(batch['labels'].cpu().numpy())

    logits = np.concatenate(all_logits, axis=0)
    labels = np.concatenate(all_labels, axis=0)
    probs  = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
    preds  = (probs >= 0.5).astype(int)

    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average='binary', zero_division=0)
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)

    try:
        auc = roc_auc_score(labels, probs)
    except ValueError:
        auc = float('nan')

    res = {
        'acc': acc,
        'precision': p,
        'recall': r,
        'f1': f1,
        'macro_precision': macro_p,
        'macro_recall': macro_r,
        'macro_f1': macro_f1,
        'auc': auc,
    }

    if return_probs:
        return res, probs, labels
    return res


def tune_threshold(val_probs, val_labels):
    best_thr, best_f1 = 0.5, -1
    rows = []

    for thr in np.linspace(0.10, 0.80, 71):
        preds = (val_probs >= thr).astype(int)
        p, r, f1, _ = precision_recall_fscore_support(
            val_labels, preds, average='binary', zero_division=0
        )
        macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
            val_labels, preds, average='macro', zero_division=0
        )
        acc = accuracy_score(val_labels, preds)

        rows.append({
            'threshold': thr,
            'accuracy': acc,
            'claim_precision': p,
            'claim_recall': r,
            'claim_f1': f1,
            'macro_f1': macro_f1,
        })

        if f1 > best_f1:
            best_f1, best_thr = f1, thr

    return best_thr, best_f1, pd.DataFrame(rows)


## 6. Multi-model training and final golden-test evaluation

## 8A. Prediction and extracted-claim JSON exports

For each model, the training function saves prediction/error review files and one combined JSON file containing extracted claims from both the LLM-labelled Dataset split and the Golden validation/test splits.

The main handoff file for T2 is `t1_extracted_claims_dataset_and_golden.json`. It is saved as a **plain JSON list** using this schema:

```json
[
  {
    "paper_id": "10.1007/s00330-024-11004-y",
    "sentence_id": 8,
    "title": "...",
    "year": "2024",
    "claim_text": "...",
    "raw_sentence": "...",
    "text": "[2024] title | sentence",
    "model": null,
    "task": null,
    "dataset": null,
    "metric": null,
    "value": null,
    "comparison": null,
    "domain": null
  }
]
```

The structured fields are intentionally `null` here because T1 only detects claim sentences. T2 will fill `model`, `task`, `dataset`, `metric`, `value`, `comparison`, and `domain`.



## 7A. Model loading note

No optional test cell is needed. Run full training after restarting the kernel.


In [9]:
# No extra loading test code. Continue to the training cell.


In [10]:

def train_one_model(cfg):
    run_name   = cfg['run_name']
    model_name = cfg['model_name']
    use_fast_tokenizer = cfg.get('use_fast_tokenizer', True)
    max_len    = cfg.get('max_len', MAX_LEN)
    batch_size = cfg.get('batch_size', BATCH_SIZE)
    epochs     = cfg.get('epochs', EPOCHS)
    lr         = cfg.get('lr', LR)

    print('\n' + '=' * 90)
    print(f'STARTING MODEL: {run_name}')
    print(f'HF checkpoint: {model_name}')
    print('=' * 90)

    # Tokenizer loading
    print(f"Tokenizer setting: use_fast_tokenizer={use_fast_tokenizer}")
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        use_fast=use_fast_tokenizer,
    )
    train_loader, val_loader, test_loader = make_loaders(tokenizer, max_len=max_len, batch_size=batch_size)

    # Simple model loading
    print("Loading model...")
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,
    )
    model.to(DEVICE)

    # label 0 = non-claim, label 1 = claim
    class_weights = torch.tensor([1.0, CLAIM_CLASS_WEIGHT], dtype=torch.float).to(DEVICE)
    loss_fn = CrossEntropyLoss(weight=class_weights)

    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(WARMUP_RATIO * total_steps),
        num_training_steps=total_steps,
    )

    best_val_f1, best_state = -1, None
    epoch_rows = []

    for epoch in range(1, epochs + 1):
        model.train()
        pbar = tqdm(train_loader, desc=f'{run_name} | Epoch {epoch}/{epochs}')
        running = 0.0

        for batch in pbar:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            out = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'])
            loss = loss_fn(out.logits, batch['labels'])

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

            running += loss.item()
            pbar.set_postfix(loss=f'{running / (pbar.n + 1):.4f}')

        val_res = evaluate(val_loader, model)
        epoch_rows.append({
            'run_name': run_name,
            'epoch': epoch,
            'val_accuracy_at_0_5': val_res['acc'],
            'val_claim_f1_at_0_5': val_res['f1'],
            'val_macro_f1_at_0_5': val_res['macro_f1'],
            'val_auc': val_res['auc'],
        })

        print(
            f'  val_acc={val_res["acc"]:.4f} '
            f'val_claim_f1@0.5={val_res["f1"]:.4f} '
            f'val_macro_f1@0.5={val_res["macro_f1"]:.4f} '
            f'val_auc={val_res["auc"]:.4f}'
        )

        if val_res['f1'] > best_val_f1:
            best_val_f1 = val_res['f1']
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            print(f'  ✓ new best validation claim F1 = {best_val_f1:.4f}')

    # reload best epoch before threshold tuning and test evaluation
    if best_state is not None:
        model.load_state_dict(best_state)
        model.to(DEVICE)

    # threshold tuning on golden validation only
    val_res, val_probs, val_labels = evaluate(val_loader, model, return_probs=True)
    best_thr, best_thr_f1, threshold_df = tune_threshold(val_probs, val_labels)
    print(f'Best threshold on golden validation = {best_thr:.3f} | claim F1 = {best_thr_f1:.4f}')

    # final test evaluation on unseen golden test
    test_res, test_probs, test_labels = evaluate(test_loader, model, return_probs=True)
    test_preds = (test_probs >= best_thr).astype(int)

    acc = accuracy_score(test_labels, test_preds)
    claim_p, claim_r, claim_f1, _ = precision_recall_fscore_support(
        test_labels, test_preds, average='binary', zero_division=0
    )
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        test_labels, test_preds, average='macro', zero_division=0
    )
    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        test_labels, test_preds, average='weighted', zero_division=0
    )
    cm = confusion_matrix(test_labels, test_preds)
    tn, fp, fn, tp = cm.ravel()

    print('\n' + '-' * 70)
    print(f'FINAL RESULT ON UNSEEN GOLDEN TEST SET: {run_name}')
    print('-' * 70)
    print(classification_report(test_labels, test_preds, target_names=['non-claim', 'claim'], digits=4))
    print('Confusion matrix [rows=true, cols=pred]:')
    print(cm)
    print(f'AUC: {test_res["auc"]:.4f}')
    print(f'Best threshold: {best_thr:.3f}')

    # save model and run artifacts
    run_dir = OUTPUT_DIR / run_name
    model_dir = MODEL_ROOT_DIR / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    model_dir.mkdir(parents=True, exist_ok=True)

    model.save_pretrained(model_dir, safe_serialization=False)
    tokenizer.save_pretrained(model_dir)

    with open(model_dir / 'threshold.json', 'w') as f:
        json.dump({'threshold': float(best_thr)}, f, indent=2)

    pd.DataFrame(epoch_rows).to_csv(run_dir / 'epoch_metrics.csv', index=False)
    threshold_df.to_csv(run_dir / 'threshold_sweep.csv', index=False)

    error_df = test_df.copy()
    error_df['split'] = 'golden_test'
    error_df['prob_claim'] = test_probs
    error_df['pred'] = test_preds
    error_df['threshold'] = float(best_thr)
    error_df['run_name'] = run_name

    # Error exports for manual review
    error_df[(error_df['label'] == 1) & (error_df['pred'] == 0)].to_csv(run_dir / 'false_negatives.csv', index=False)
    error_df[(error_df['label'] == 0) & (error_df['pred'] == 1)].to_csv(run_dir / 'false_positives.csv', index=False)

    # Core prediction export on the unseen golden test set
    # Note: claim handoff files are saved as JSON below, not CSV.
    error_df.to_csv(run_dir / 'golden_test_predictions.csv', index=False)

    # ------------------------------------------------------------------
    # Save extracted claims from Dataset + Golden splits in ONE JSON file
    # for downstream T2/T3/T4 research stages.
    # ------------------------------------------------------------------
    @torch.no_grad()
    def predict_dataframe_for_export(df, split_name):
        export_ds = ClaimDataset(df, tokenizer, max_len=max_len)
        export_loader = DataLoader(
            export_ds,
            batch_size=batch_size * 2,
            shuffle=False,
            num_workers=2,
        )
        _, probs, labels = evaluate(export_loader, model, return_probs=True)
        preds = (probs >= best_thr).astype(int)

        out_df = df.copy().reset_index(drop=True)
        out_df['split'] = split_name
        out_df['prob_claim'] = probs
        out_df['pred'] = preds
        out_df['threshold'] = float(best_thr)
        out_df['run_name'] = run_name

        # Add a simple error type where gold labels exist.
        conditions = [
            (out_df['label'] == 1) & (out_df['pred'] == 1),
            (out_df['label'] == 0) & (out_df['pred'] == 0),
            (out_df['label'] == 0) & (out_df['pred'] == 1),
            (out_df['label'] == 1) & (out_df['pred'] == 0),
        ]
        choices = ['TP', 'TN', 'FP', 'FN']
        out_df['prediction_type'] = np.select(conditions, choices, default='unknown')
        return out_df

    if SAVE_ALL_EXTRACTED_CLAIMS:
        export_frames = []

        if EXPORT_TRAIN_PREDICTIONS:
            train_pred_export = predict_dataframe_for_export(train_df, 'dataset_train')
            # Save full predictions as CSV only for debugging/reproducibility.
            # Extracted claims themselves are saved in one JSON below.
            train_pred_export.to_csv(run_dir / 'dataset_train_predictions.csv', index=False)
            export_frames.append(train_pred_export)

        val_pred_export = predict_dataframe_for_export(val_df, 'golden_validation')
        val_pred_export.to_csv(run_dir / 'golden_validation_predictions.csv', index=False)
        export_frames.append(val_pred_export)

        test_pred_export = error_df.copy()
        test_pred_export['prediction_type'] = np.select(
            [
                (test_pred_export['label'] == 1) & (test_pred_export['pred'] == 1),
                (test_pred_export['label'] == 0) & (test_pred_export['pred'] == 0),
                (test_pred_export['label'] == 0) & (test_pred_export['pred'] == 1),
                (test_pred_export['label'] == 1) & (test_pred_export['pred'] == 0),
            ],
            ['TP', 'TN', 'FP', 'FN'],
            default='unknown',
        )
        export_frames.append(test_pred_export)

        all_predictions_df = pd.concat(export_frames, ignore_index=True)
        all_extracted_claims_df = all_predictions_df[all_predictions_df['pred'] == 1].copy()

        # Full predictions can remain CSV for checking model behavior.
        all_predictions_df.to_csv(run_dir / 'all_predictions.csv', index=False)

        # Clean JSON handoff for T2 structured extraction.
        # The JSON follows the exact claim schema used in the research pipeline:
        # paper_id, sentence_id, title, year, claim_text, raw_sentence, text,
        # model, task, dataset, metric, value, comparison, domain.
        claim_text_col = 'sentence' if 'sentence' in all_extracted_claims_df.columns else 'input'
        all_extracted_claims_df = all_extracted_claims_df.reset_index(drop=True)

        def none_if_missing(value):
            if pd.isna(value):
                return None
            return value

        def get_row_value(row, candidates, default=None):
            for col in candidates:
                if col in row.index and not pd.isna(row[col]):
                    return row[col]
            return default

        claims_records = []
        for i, row in all_extracted_claims_df.iterrows():
            paper_id = get_row_value(row, ['paper_id', 'doi'], default=None)
            title = get_row_value(row, ['title'], default=None)
            year = get_row_value(row, ['year'], default=None)
            raw_sentence = get_row_value(row, ['raw_sentence', 'sentence', 'input'], default='')
            claim_text = get_row_value(row, ['claim_text', 'raw_sentence', 'sentence', 'input'], default=raw_sentence)

            if 'sentence_id' in row.index and not pd.isna(row['sentence_id']):
                sentence_id = int(row['sentence_id']) if isinstance(row['sentence_id'], (int, np.integer, float, np.floating)) else row['sentence_id']
            else:
                # If the original dataset has no sentence_id, create a stable 1-based id inside this export.
                sentence_id = int(i + 1)

            # Full context text used by the context-based classifier and T2 handoff.
            text_parts = []
            if year is not None:
                text_parts.append(f'[{year}]')
            if title is not None:
                text_parts.append(str(title))
            if claim_text is not None:
                text_parts.append(str(claim_text))
            text = ' | '.join(text_parts)

            claims_records.append({
                'paper_id': none_if_missing(paper_id),
                'sentence_id': sentence_id,
                'title': none_if_missing(title),
                'year': str(int(year)) if isinstance(year, (int, np.integer, float, np.floating)) and not pd.isna(year) else none_if_missing(year),
                'claim_text': none_if_missing(claim_text),
                'raw_sentence': none_if_missing(raw_sentence),
                'text': text,
                # Empty structured fields for T2. These will be filled by the structured extraction model.
                'model': None,
                'task': None,
                'dataset': None,
                'metric': None,
                'value': None,
                'comparison': None,
                'domain': None,
            })

        combined_json_path = run_dir / 't1_extracted_claims_dataset_and_golden.json'
        with open(combined_json_path, 'w', encoding='utf-8') as f:
            json.dump(claims_records, f, ensure_ascii=False, indent=2)

        print(f'Saved all predictions: {run_dir / "all_predictions.csv"}')
        print(f'Saved combined extracted claims JSON: {combined_json_path}')
        print(f'Total extracted claims in JSON: {len(claims_records)}')

    summary = {
        'run_name': run_name,
        'model_name': model_name,
        'max_len': max_len,
        'batch_size': batch_size,
        'epochs': epochs,
        'claim_class_weight': CLAIM_CLASS_WEIGHT,
        'best_threshold': best_thr,
        'val_best_claim_f1': best_thr_f1,
        'test_accuracy': acc,
        'test_claim_precision': claim_p,
        'test_claim_recall': claim_r,
        'test_claim_f1': claim_f1,
        'test_macro_precision': macro_p,
        'test_macro_recall': macro_r,
        'test_macro_f1': macro_f1,
        'test_weighted_f1': weighted_f1,
        'test_auc': test_res['auc'],
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'tp': int(tp),
    }

    # cleanup GPU memory before next model
    del model, tokenizer, train_loader, val_loader, test_loader
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return summary


all_results = []
for cfg in MODEL_CONFIGS:
    try:
        result = train_one_model(cfg)
        all_results.append(result)
        pd.DataFrame(all_results).to_csv(OUTPUT_DIR / 't1_multi_model_results.csv', index=False)
    except Exception as e:
        print(f'✗ Failed model {cfg["run_name"]}: {e}')
        all_results.append({
            'run_name': cfg['run_name'],
            'model_name': cfg['model_name'],
            'error': str(e),
        })
        pd.DataFrame(all_results).to_csv(OUTPUT_DIR / 't1_multi_model_results.csv', index=False)

results_df = pd.DataFrame(all_results)
results_df




STARTING MODEL: SciBERT_Context
HF checkpoint: allenai/scibert_scivocab_uncased
Tokenizer setting: use_fast_tokenizer=True
Loading model...


2026-05-11 05:30:07.543437: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-11 05:30:07.543473: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-11 05:30:07.545077: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-11 05:30:07.554244: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-11 05:30:08.856167: W tensorflow/compiler/tf2

SciBERT_Context | Epoch 1/4:   0%|          | 0/860 [00:00<?, ?it/s]

  val_acc=0.8106 val_claim_f1@0.5=0.7250 val_macro_f1@0.5=0.7903 val_auc=0.9026
  ✓ new best validation claim F1 = 0.7250


SciBERT_Context | Epoch 2/4:   0%|          | 0/860 [00:00<?, ?it/s]

  val_acc=0.8117 val_claim_f1@0.5=0.7592 val_macro_f1@0.5=0.8023 val_auc=0.8984
  ✓ new best validation claim F1 = 0.7592


... (13 outputs left out)

PubMedBERT_Context | Epoch 4/4:   0%|          | 0/860 [00:00<?, ?it/s]

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



  val_acc=0.8094 val_claim_f1@0.5=0.7418 val_macro_f1@0.5=0.7954 val_auc=0.8908
Best threshold on golden validation = 0.650 | claim F1 = 0.7886

----------------------------------------------------------------------
FINAL RESULT ON UNSEEN GOLDEN TEST SET: BioBERT_Context
----------------------------------------------------------------------
              precision    recall  f1-score   support

   non-claim     0.8456    0.8755    0.8603      1020
       claim     0.8304    0.7924    0.8110       785

    accuracy                         0.8393      1805
   macro avg     0.8380    0.8339    0.8356      1805
weighted avg     0.8390    0.8393    0.8388      1805

Confusion matrix [rows=true, cols=pred]:
[[893 127]
 [163 622]]
AUC: 0.9265
Best threshold: 0.650
Saved all predictions: outputs_multi_model/BioBERT_Context/all_predictions.csv
Saved combined extracted claims JSON: outputs_multi_model/BioBERT_Context/t1_extracted_claims_dataset_and_golden.json
Total extracted claims in JSON: 759

,run_name,model_name,max_len,batch_size,epochs,claim_class_weight,best_threshold,val_best_claim_f1,test_accuracy,test_claim_precision,...,test_macro_precision,test_macro_recall,test_macro_f1,test_weighted_f1,test_auc,tn,fp,fn,tp,error
0,SciBERT_Context,allenai/scibert_scivocab_uncased,256.0,16.0,4.0,3.0,0.17,0.773762,0.831025,0.815789,...,0.828947,0.826277,0.827445,0.830681,0.918182,880.0,140.0,165.0,620.0,NaN
1,PubMedBERT_Context,microsoft/BiomedNLP-PubMedBERT-base-uncased-ab...,256.0,16.0,4.0,3.0,0.56,0.783964,0.826039,0.880452,...,0.839046,0.810859,0.817007,0.822300,0.924795,946.0,74.0,240.0,545.0,NaN
2,BioBERT_Context,monologg/biobert_v1.1_pubmed,256.0,16.0,4.0,3.0,0.65,0.788618,0.839335,0.830441,...,0.838042,0.833923,0.835630,0.838843,0.926488,893.0,127.0,163.0,622.0,NaN
3,ModernBERT_Context,answerdotai/ModernBERT-base,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,The checkpoint you are trying to load has mode...


## 7. Leaderboard

In [11]:

leaderboard = pd.DataFrame(all_results)
metric_cols = [
    'run_name', 'test_accuracy', 'test_macro_f1', 'test_claim_f1',
    'test_claim_precision', 'test_claim_recall', 'test_auc',
    'best_threshold', 'tn', 'fp', 'fn', 'tp'
]
existing_cols = [c for c in metric_cols if c in leaderboard.columns]
leaderboard = leaderboard[existing_cols].sort_values('test_macro_f1', ascending=False)

print('MULTI-MODEL LEADERBOARD')
display(leaderboard)
leaderboard.to_csv(OUTPUT_DIR / 't1_multi_model_leaderboard.csv', index=False)
print('Saved:', OUTPUT_DIR / 't1_multi_model_leaderboard.csv')


MULTI-MODEL LEADERBOARD


,run_name,test_accuracy,test_macro_f1,test_claim_f1,test_claim_precision,test_claim_recall,test_auc,best_threshold,tn,fp,fn,tp
2,BioBERT_Context,0.839335,0.835630,0.810952,0.830441,0.792357,0.926488,0.65,893.0,127.0,163.0,622.0
0,SciBERT_Context,0.831025,0.827445,0.802589,0.815789,0.789809,0.918182,0.17,880.0,140.0,165.0,620.0
1,PubMedBERT_Context,0.826039,0.817007,0.776353,0.880452,0.694268,0.924795,0.56,946.0,74.0,240.0,545.0
3,ModernBERT_Context,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Saved: outputs_multi_model/t1_multi_model_leaderboard.csv


## 8. Save best T1 claim-detection model and optionally push to Hugging Face Hub

This section selects the best model from the multi-model leaderboard, saves the model/tokenizer locally, creates a research-specific model card, and optionally uploads the model to Hugging Face Hub.

Default selection metric: **test_macro_f1** on the unseen manually annotated golden test set.

In [12]:

# ==========================================================================
# SAVE BEST MULTI-MODEL T1 CLAIM DETECTOR & PUSH TO HUGGING FACE HUB
# ==========================================================================

from huggingface_hub import HfApi
from datetime import datetime
from getpass import getpass
import shutil
import json

print("=" * 80)
print("T1 CLAIM DETECTION: SAVE BEST MODEL & PUSH TO HUGGING FACE HUB")
print("=" * 80)

# --------------------------------------------------------------------------
# 1. Select best model from leaderboard
# --------------------------------------------------------------------------
if "leaderboard" not in globals() or leaderboard.empty:
    leaderboard_path = OUTPUT_DIR / "t1_multi_model_leaderboard.csv"
    if leaderboard_path.exists():
        leaderboard = pd.read_csv(leaderboard_path)
    else:
        raise FileNotFoundError(
            "Leaderboard not found. Run the multi-model training and leaderboard cells first."
        )

# Choose the model by macro F1 because this is the fairest metric for claim/non-claim balance.
BEST_SELECTION_METRIC = "test_macro_f1"

valid_leaderboard = leaderboard.dropna(subset=[BEST_SELECTION_METRIC]).copy()
if valid_leaderboard.empty:
    raise ValueError(f"No valid rows found for metric: {BEST_SELECTION_METRIC}")

best_row = valid_leaderboard.sort_values(BEST_SELECTION_METRIC, ascending=False).iloc[0]
best_run_name = best_row["run_name"]

# Retrieve the original HF checkpoint name from all_results / results csv.
results_path = OUTPUT_DIR / "t1_multi_model_results.csv"
if "results_df" in globals() and not results_df.empty:
    results_lookup_df = results_df.copy()
elif results_path.exists():
    results_lookup_df = pd.read_csv(results_path)
else:
    raise FileNotFoundError("t1_multi_model_results.csv not found.")

best_result_row = results_lookup_df[results_lookup_df["run_name"] == best_run_name].iloc[0]
base_model_name = best_result_row["model_name"]

best_model_dir = MODEL_ROOT_DIR / best_run_name
if not best_model_dir.exists():
    raise FileNotFoundError(f"Saved model directory not found: {best_model_dir}")

print(f"Best model selected by {BEST_SELECTION_METRIC}: {best_run_name}")
print(f"Base checkpoint: {base_model_name}")
print(f"Saved model directory: {best_model_dir}")

# --------------------------------------------------------------------------
# 2. Create clean local export directory
# --------------------------------------------------------------------------
EXPORT_ROOT = OUTPUT_DIR / "huggingface_export"
HF_EXPORT_DIR = EXPORT_ROOT / best_run_name
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

if HF_EXPORT_DIR.exists():
    shutil.rmtree(HF_EXPORT_DIR)
shutil.copytree(best_model_dir, HF_EXPORT_DIR)

# --------------------------------------------------------------------------
# 3. Collect metrics for model card
# --------------------------------------------------------------------------
def fmt_metric(x):
    try:
        return f"{float(x):.4f}"
    except Exception:
        return "N/A"

metrics = {
    "accuracy": fmt_metric(best_row.get("test_accuracy")),
    "macro_f1": fmt_metric(best_row.get("test_macro_f1")),
    "claim_f1": fmt_metric(best_row.get("test_claim_f1")),
    "claim_precision": fmt_metric(best_row.get("test_claim_precision")),
    "claim_recall": fmt_metric(best_row.get("test_claim_recall")),
    "auc": fmt_metric(best_row.get("test_auc")),
    "threshold": fmt_metric(best_row.get("best_threshold")),
    "tn": int(best_row.get("tn", 0)) if pd.notna(best_row.get("tn", None)) else "N/A",
    "fp": int(best_row.get("fp", 0)) if pd.notna(best_row.get("fp", None)) else "N/A",
    "fn": int(best_row.get("fn", 0)) if pd.notna(best_row.get("fn", None)) else "N/A",
    "tp": int(best_row.get("tp", 0)) if pd.notna(best_row.get("tp", None)) else "N/A",
}

# Build a compact ablation/leaderboard table for README.
leaderboard_cols = [
    "run_name", "test_accuracy", "test_macro_f1", "test_claim_f1",
    "test_claim_precision", "test_claim_recall", "test_auc", "best_threshold"
]
leaderboard_table_df = leaderboard[[c for c in leaderboard_cols if c in leaderboard.columns]].copy()
for c in leaderboard_table_df.columns:
    if c != "run_name":
        leaderboard_table_df[c] = leaderboard_table_df[c].apply(fmt_metric)
leaderboard_md = leaderboard_table_df.to_markdown(index=False)

# --------------------------------------------------------------------------
# 4. Create research-specific model card
# --------------------------------------------------------------------------
model_card = f"""---
library_name: transformers
pipeline_tag: text-classification
license: mit
language:
- en
tags:
- scientific-claim-detection
- biomedical-nlp
- brain-tumor
- scibert
- pubmedbert
- biobert
- modernbert
- text-classification
---

# T1 Claim Detection Model for Brain Tumor Research Abstracts

This repository contains the best T1 claim-detection model selected from a multi-model experiment for the thesis research project:

**Automated Scientific Claim Extraction, Comparison, and Knowledge Graph Construction from Brain Tumor Detection Literature Using Transformer-Based Models**.

The model classifies abstract sentences from brain tumor research papers into:

- `0`: non-claim
- `1`: claim

A claim is defined as a sentence that reports a result, finding, performance value, treatment effect, association, comparison, conclusion, recommendation, or evidence-based scientific statement.

## Best Model

- **Run name:** `{best_run_name}`
- **Base checkpoint:** `{base_model_name}`
- **Selection metric:** `{BEST_SELECTION_METRIC}`
- **Decision threshold:** `{metrics['threshold']}`

## Final Performance on Unseen Golden Test Set

The test set was manually annotated and kept unseen during training and threshold tuning.

| Metric | Score |
|---|---:|
| Accuracy | {metrics['accuracy']} |
| Macro F1 | {metrics['macro_f1']} |
| Claim Precision | {metrics['claim_precision']} |
| Claim Recall | {metrics['claim_recall']} |
| Claim F1 | {metrics['claim_f1']} |
| AUC | {metrics['auc']} |

### Confusion Matrix

Rows are true labels and columns are predicted labels.

|  | Pred non-claim | Pred claim |
|---|---:|---:|
| True non-claim | {metrics['tn']} | {metrics['fp']} |
| True claim | {metrics['fn']} | {metrics['tp']} |

## Dataset and Evaluation Design

- Domain: brain tumor research abstracts
- Total corpus: 1,586 papers
- Golden/manual subset: 316 papers
- Training: non-golden LLM-labelled papers
- Validation: manually annotated golden validation papers
- Test: unseen manually annotated golden test papers
- Split level: paper-level split to avoid DOI/paper leakage
- Threshold tuning: performed only on golden validation, not on golden test

## Multi-Model Comparison

{leaderboard_md}

## Intended Use

This model is intended for research use in a scientific claim-mining pipeline. It can be used as the first stage for identifying claim sentences before structured extraction of fields such as model, task, dataset, metric, value, comparison, and domain.

## Limitations

- The model was trained for brain tumor research abstracts and may not generalize directly to unrelated biomedical domains.
- It may confuse methods/objectives with claims when a sentence is written in a result-like style.
- It may miss implicit claims that do not contain clear result or conclusion signals.
- It should not be used as a clinical decision-making tool.

## Example Usage

```python
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

repo_id = "YOUR_USERNAME/YOUR_REPO_NAME"
threshold = {metrics['threshold']}

tokenizer = AutoTokenizer.from_pretrained(repo_id)
model = AutoModelForSequenceClassification.from_pretrained(repo_id)
model.eval()

sentence = "The proposed model achieved a Dice score of 0.91 on the BraTS 2020 dataset."
inputs = tokenizer(sentence, return_tensors="pt", truncation=True, padding=True, max_length=256)

with torch.no_grad():
    logits = model(**inputs).logits
    prob_claim = torch.softmax(logits, dim=-1)[0, 1].item()

prediction = int(prob_claim >= threshold)
print({{"prob_claim": prob_claim, "prediction": prediction}})
```

## Training Details

- Architecture: `AutoModelForSequenceClassification`
- Loss: weighted cross-entropy
- Positive claim class weight: `{CLAIM_CLASS_WEIGHT}`
- Context input: enabled
- Optimizer: AdamW
- Learning rate: `{LR}`
- Max sequence length: `{MAX_LEN}`
- Epochs per model: `{EPOCHS}`
- Export date: `{datetime.now().isoformat()}`
"""

(HF_EXPORT_DIR / "README.md").write_text(model_card, encoding="utf-8")

# Save extra metadata for reproducibility.
export_config = {
    "best_run_name": best_run_name,
    "base_model_name": base_model_name,
    "selection_metric": BEST_SELECTION_METRIC,
    "metrics": metrics,
    "claim_class_weight": CLAIM_CLASS_WEIGHT,
    "use_context": True,
    "max_len": MAX_LEN,
    "epochs": EPOCHS,
    "learning_rate": LR,
    "warmup_ratio": WARMUP_RATIO,
    "export_date": datetime.now().isoformat(),
}
(HF_EXPORT_DIR / "research_training_config.json").write_text(
    json.dumps(export_config, indent=2), encoding="utf-8"
)

print("\n✓ Hugging Face export folder prepared:", HF_EXPORT_DIR)
print("Files:")
for f in sorted(HF_EXPORT_DIR.iterdir()):
    print(" -", f.name)

# --------------------------------------------------------------------------
# 5. Optional push to Hugging Face Hub
# --------------------------------------------------------------------------
print("\n" + "=" * 80)
print("OPTIONAL: PUSH TO HUGGING FACE HUB")
print("=" * 80)
print("You need a Hugging Face access token with write permission.")

PUSH_TO_HUB = input("Push this best model to Hugging Face Hub? (yes/no, default=no): ").strip().lower() == "yes"

if PUSH_TO_HUB:
    HF_TOKEN = getpass("Enter your Hugging Face token (hidden): ").strip()
    if not HF_TOKEN.startswith("hf_"):
        raise ValueError("Invalid Hugging Face token. It should start with 'hf_'.")

    HF_USERNAME = input("Enter your Hugging Face username or organization: ").strip()
    if not HF_USERNAME:
        raise ValueError("Hugging Face username/organization cannot be empty.")

    default_repo_name = f"brain-tumor-claim-detector-{best_run_name.lower().replace('_', '-')}"
    REPO_NAME = input(f"Repository name [default: {default_repo_name}]: ").strip() or default_repo_name
    PRIVATE_REPO = input("Make repository private? (yes/no, default=no): ").strip().lower() == "yes"

    repo_id = f"{HF_USERNAME}/{REPO_NAME}"
    api = HfApi()

    print(f"\nCreating/updating repository: {repo_id}")
    api.create_repo(
        repo_id=repo_id,
        repo_type="model",
        private=PRIVATE_REPO,
        exist_ok=True,
        token=HF_TOKEN,
    )

    print("Uploading model files...")
    api.upload_folder(
        folder_path=str(HF_EXPORT_DIR),
        repo_id=repo_id,
        repo_type="model",
        token=HF_TOKEN,
        commit_message=(
            f"Upload best T1 claim detector: {best_run_name} "
            f"macroF1={metrics['macro_f1']} claimF1={metrics['claim_f1']}"
        ),
    )

    print("\n✅ Upload complete")
    print(f"Model URL: https://huggingface.co/{repo_id}")
    print(f"Use with: AutoModelForSequenceClassification.from_pretrained('{repo_id}')")
else:
    print("\nSkipping Hub upload.")
    print(f"Local export remains available at: {HF_EXPORT_DIR}")


T1 CLAIM DETECTION: SAVE BEST MODEL & PUSH TO HUGGING FACE HUB
Best model selected by test_macro_f1: BioBERT_Context
Base checkpoint: monologg/biobert_v1.1_pubmed
Saved model directory: models/t1_multi_model_claim_detectors/BioBERT_Context

✓ Hugging Face export folder prepared: outputs_multi_model/huggingface_export/BioBERT_Context
Files:
 - README.md
 - config.json
 - pytorch_model.bin
 - research_training_config.json
 - special_tokens_map.json
 - threshold.json
 - tokenizer_config.json
 - vocab.txt

OPTIONAL: PUSH TO HUGGING FACE HUB
You need a Hugging Face access token with write permission.


Push this best model to Hugging Face Hub? (yes/no, default=no):  yes
Enter your Hugging Face token (hidden):  ········
Enter your Hugging Face username or organization:  nawazishpatana
Repository name [default: brain-tumor-claim-detector-biobert-context]:  brain-tumor-claim-detector-biobert-context
Make repository private? (yes/no, default=no):  yes



Creating/updating repository: nawazishpatana/brain-tumor-claim-detector-biobert-context
Uploading model files...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


✅ Upload complete
Model URL: https://huggingface.co/nawazishpatana/brain-tumor-claim-detector-biobert-context
Use with: AutoModelForSequenceClassification.from_pretrained('nawazishpatana/brain-tumor-claim-detector-biobert-context')
